In [ ]:
import time
import json
import websocket
import socket
import numpy as np
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds

# ─── CONFIGURATION ───────────────────────────────────────────────────────────
# Set USE_SIMULATOR=True for development/testing without hardware
# Set USE_SIMULATOR=False and configure SERIAL_PORT for real Cyton+Daisy headset
USE_SIMULATOR = False
SERIAL_PORT = "COM10"
SERVER_URI = "ws://127.0.0.1:65432"  # Change to PC's IP for laptop→PC streaming

CHUNK_SIZE = 16          # Send 16 samples at a time (~128ms at 125Hz)
SEND_INTERVAL = 0.016    # ~60Hz send rate (smooth UI updates)

def main():
    BoardShim.enable_dev_board_logger()
    params = BrainFlowInputParams()

    board_id = BoardIds.SYNTHETIC_BOARD if USE_SIMULATOR else BoardIds.CYTON_DAISY_BOARD
    if not USE_SIMULATOR:
        params.serial_port = SERIAL_PORT

    board = BoardShim(board_id, params)
    ws = None

    try:
        board.prepare_session()
        board.start_stream(4500)  # Ring buffer: ~36s at 125Hz
        print(f"✅ Stream started ({'SIMULATOR' if USE_SIMULATOR else 'Cyton+Daisy'})")
        print("   Waiting 3s for signal stabilization...")
        time.sleep(3)

        eeg_channels = BoardShim.get_eeg_channels(board_id)[:16]
        sample_rate = BoardShim.get_sampling_rate(board_id)
        print(f"ℹ️  EEG channels: {len(eeg_channels)} | Sample rate: {sample_rate} Hz")

        ws = websocket.WebSocket()
        ws.settimeout(5)
        print(f"🔌 Connecting to {SERVER_URI}...")
        ws.connect(SERVER_URI, timeout=10)
        print("✅ CONNECTED to inference server!\n")

        # Accumulation buffer for smooth fixed-size sends
        accum_buffer = np.zeros((16, 0))
        chunks_sent = 0
        last_send_time = time.perf_counter()

        while True:
            # Get whatever the board has buffered (non-blocking)
            data = board.get_board_data()

            if data.shape[1] > 0:
                eeg_data = data[eeg_channels, :]
                accum_buffer = np.concatenate((accum_buffer, eeg_data), axis=1)

            # Send in fixed-size chunks for smooth inference
            now = time.perf_counter()
            while accum_buffer.shape[1] >= CHUNK_SIZE:
                chunk = accum_buffer[:, :CHUNK_SIZE]
                accum_buffer = accum_buffer[:, CHUNK_SIZE:]

                try:
                    ws.send(json.dumps(chunk.tolist()))
                    chunks_sent += 1
                except Exception as e:
                    print(f"\n⚠️ Send error: {e}")
                    try:
                        ws.close()
                        time.sleep(1)
                        ws = websocket.WebSocket()
                        ws.settimeout(5)
                        ws.connect(SERVER_URI, timeout=10)
                        print("🔄 Reconnected!")
                    except:
                        pass
                    break

            # Non-blocking drain (single pass, no while loop)
            try:
                if ws.sock:
                    ws.sock.settimeout(0)
                    ws.recv()
            except:
                pass

            # Status update
            if chunks_sent % 200 == 0 and chunks_sent > 0:
                print(f"   📤 Chunks: {chunks_sent} | Buffer: {accum_buffer.shape[1]} samples", end='\r')

            # Pace the loop — short sleep to avoid CPU spin while keeping latency low
            elapsed = time.perf_counter() - now
            sleep_time = max(0, SEND_INTERVAL - elapsed)
            if sleep_time > 0:
                time.sleep(sleep_time)

    except KeyboardInterrupt:
        print("\n\n🛑 Manual stop.")
    finally:
        if ws:
            ws.close()
        if board.is_prepared():
            board.release_session()
            print("🔌 Hardware session released.")

if __name__ == "__main__":
    main()

✅ Stream started (Cyton+Daisy)
   Waiting 3s for signal stabilization...
ℹ️  EEG channels: 16 | Sample rate: 125 Hz
🔌 Connecting to ws://127.0.0.1:65432...
✅ CONNECTED to inference server!

   📤 Chunks: 800 | Buffer: 13 samples
⚠️ Send error: [WinError 10054] An existing connection was forcibly closed by the remote host
